# Massive Activations in FLUX — Colab runner

Two experiments on the same capture/ranking core:

- **Part 1 — Figure 3 (FLUX.2-klein).** Reproduce Figure 3 / Section 3.2 of *"Few Channels
  Draw The Whole Picture: Revealing Massive Activations in Diffusion Transformers"*
  (arXiv:2605.13974): top-/bottom-/random-k channel masks vs. a BiRefNet pseudo-GT, and the
  layer-wise mIoU curve (Fig 3D).
- **Part 2 — Per-generation channel stability (FLUX.1-dev).** Adjacent experiment: for each
  (prompt, seed) generation, at a fixed block + timestep, rank channels by the
  massive-activation score and measure how much the top-channel *identities* change across
  generations. See `src/experiments/channel_stability.py`.

Sections 1–4 (setup, install, HF auth) are shared. Run Part 1 (§5–8) and/or Part 2 (§9–11)
independently; §12 downloads results.

**Before you run:**
1. `Runtime > Change runtime type` → GPU. T4 works for Part 1; **Part 2 uses the 12B
   FLUX.1-dev — prefer A100/L4** (it falls back to CPU-offload on smaller GPUs, slow).
2. Hugging Face account + `HF_TOKEN` (Colab left sidebar → 🔑 *Secrets*). **Accept the model
   licenses** for the checkpoints you'll use: `black-forest-labs/FLUX.2-klein`,
   `ZhengPeng7/BiRefNet` (Part 1) and the gated `black-forest-labs/FLUX.1-dev` (Part 2).
3. Part 1's full run needs the 1,600 GenAI-Bench prompts (`BaiqiL/GenAI-Bench` or a local
   file); a tiny built-in prompt list is used for the smoke test first.
4. Mount Google Drive (below) so outputs survive a disconnect — Part 1 is resumable.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader \
    || echo "No GPU detected — set Runtime > Change runtime type > GPU"
import sys

print(sys.version)

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/BrendanGho/massive-activations-fig3.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
REPO_DIR = "/content/massive-activations-fig3"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — pulling latest {BRANCH}...")
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}

## 2. (Optional) Mount Google Drive for persistent storage

Recommended for the full 1,600-prompt run — writes then survive a session ending
mid-run. Skip this and outputs just live in the (ephemeral) Colab VM instead.

In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/figure3_repro"
else:
    DRIVE_ROOT = "/content/figure3_repro"

OUTPUT_DIR = f"{DRIVE_ROOT}/outputs"
CACHE_DIR = f"{DRIVE_ROOT}/cache"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("output_dir:", OUTPUT_DIR)
print("activation_cache_dir:", CACHE_DIR)

## 3. Install dependencies

Editable install of this repo plus the `fig3` extra (torch / diffusers / transformers /
scikit-learn / matplotlib / ...). Covers **both** parts — no extra install for Part 2.
Takes a few minutes on a fresh Colab runtime.

In [ ]:
!pip install -q -e ".[fig3]"

In [ ]:
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, _minor = torch.cuda.get_device_capability(0)
    DEVICE = "cuda"
    # bf16 needs Ampere+ (compute capability >= 8, e.g. A100/L4); T4 (cc 7.5) falls back to fp16.
    DTYPE = "bf16" if major >= 8 else "fp16"
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        "| compute capability:",
        major,
        "| using dtype:",
        DTYPE,
    )
else:
    DEVICE = "cpu"
    DTYPE = "fp32"
    print("No GPU — falling back to CPU/fp32. This will be extremely slow; use a GPU runtime.")

## 4. Hugging Face authentication

The FLUX checkpoints and BiRefNet may be gated. Add an `HF_TOKEN` under Colab's 🔑 *Secrets*
panel (recommended, persists across sessions), or log in interactively below. Make sure
you've accepted the license on each model page you intend to use (see the intro).

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token)
    print("Logged in via Colab secret HF_TOKEN.")
else:
    login()  # interactive prompt

# Part 1 — Figure 3 (FLUX.2-klein)

Reproduce the qualitative masks (Fig 3A–C) and the layer-wise mIoU curve (Fig 3D). Skip to
§9 if you only want the channel-stability experiment.

## 5. Configure the run

Fills the config loader's required keys (`src/common/config.py`) and writes a
Colab-specific `configs/colab.yaml` — the committed `configs/default.yaml` stays untouched.
`PROMPT_SOURCE` is the one value you should repoint at real data before the full run
(section 7); it's left as the smoke-test file until then. (The 1,600 GenAI-Bench prompts
live at the HF dataset id `BaiqiL/GenAI-Bench`.)

In [ ]:
import yaml

MODEL_CKPT = "black-forest-labs/FLUX.2-klein"  # @param {type:"string"}
BIREFNET_WEIGHTS = "ZhengPeng7/BiRefNet"  # @param {type:"string"}
# 1,600 GenAI-Bench prompts: point this at a .txt/.json/.jsonl/.parquet file or the HF
# dataset id "BaiqiL/GenAI-Bench" before the full run (section 7). Left as the smoke-test
# file until then.
PROMPT_SOURCE = f"{OUTPUT_DIR}/smoke_test_prompts.txt"  # @param {type:"string"}

# A handful of prompts so stage1/stage4 can be sanity-checked before the full run.
_SMOKE_PROMPTS = [
    "a red bicycle leaning against a brick wall",
    "a golden retriever sitting on a wooden dock at sunset",
    "a bowl of fresh strawberries on a marble countertop",
    "an astronaut planting a flag on the moon",
]
if not os.path.isfile(PROMPT_SOURCE):
    with open(PROMPT_SOURCE, "w") as fh:
        fh.write("\n".join(_SMOKE_PROMPTS) + "\n")

config = {
    "model_ckpt": MODEL_CKPT,
    "prompt_source": PROMPT_SOURCE,
    "birefnet_weights": BIREFNET_WEIGHTS,
    "output_dir": OUTPUT_DIR,
    "activation_cache_dir": CACHE_DIR,
    "num_denoising_steps": 4,
    "resolution": 1024,
    "top_k": 12,
    "layers": "all",
    "random_k_trials": 5,
    "seed": 0,
    "device": DEVICE,
    "dtype": DTYPE,
    "batch_size": 1,
    "num_example_prompts": 8,
    "cache_batch_size": 32,
    "guidance_scale": None,
}

CONFIG_PATH = "configs/colab.yaml"
with open(CONFIG_PATH, "w") as fh:
    yaml.safe_dump(config, fh, sort_keys=False)

print(open(CONFIG_PATH).read())

## 6. Quick smoke test

Runs the 4 built-in prompts through both stages end to end — catches config, auth, or
model-loading issues in minutes instead of partway through the full 1,600-prompt run.

In [ ]:
!python -m src.stage1_generate_and_cache --config {CONFIG_PATH} --fused --limit 4
!python -m src.stage4_evaluate_figure3d --config {CONFIG_PATH}

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{OUTPUT_DIR}/figure3d_curve.png"))

## 7. Full run (1,600 GenAI-Bench prompts)

Point `PROMPT_SOURCE` in section 5 at the real prompt set, re-run that cell to regenerate
`configs/colab.yaml`, then run the full resumable pipeline below. This is the expensive
step — budget real GPU time. If the session disconnects mid-run, just re-run this cell:
`stage1` skips prompts already in `activation_cache_dir`/`completed.json`.

In [ ]:
!bash scripts/run_pipeline.sh {CONFIG_PATH}

## 8. Results

In [ ]:
import glob

import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{OUTPUT_DIR}/figure3d_curve.png"))

for qdir in sorted(glob.glob(f"{OUTPUT_DIR}/qualitative/prompt_*"))[:3]:
    print(qdir)
    for png in sorted(glob.glob(f"{qdir}/*.png"))[:4]:
        display(Image(filename=png, width=220))

# Part 2 — Per-generation channel stability (FLUX.1-dev)

*"For each individual image generation, which channels are the largest, and do they change
depending on what the model is generating / from one generation to the next?"*

We fix **one block (layer 11)** and the **last denoising step**, run a cross-product of
`prompts × seeds` **scenarios** on **FLUX.1-dev**, rank channels independently per scenario
by `abs(mean_over_tokens)`, and analyze how stable the top-channel identities are. Outputs:
ordered top-20 per scenario, a scenario × channel table, a `stability_summary.json`
(same-prompt/different-seed vs different-prompt overlap), and per-scenario qualitative dumps
(image, per-channel spatial maps, aggregated top-k heatmap).

Requires §1–4 above (repo, install, HF auth). **FLUX.1-dev is gated + 12B** — accept its
license and prefer an A100/L4; on smaller GPUs it auto-enables CPU offload (slow).

## 9. Configure the channel-stability run

In [ ]:
import yaml

CS_MODEL_CKPT = "black-forest-labs/FLUX.1-dev"  # @param {type:"string"}
CS_FIXED_LAYER = 11  # @param {type:"integer"}
CS_OUTPUT_DIR = f"{DRIVE_ROOT}/channel_stability"
os.makedirs(CS_OUTPUT_DIR, exist_ok=True)

# 12B in bf16 ≈ 24 GB. Offload unless we detect a large GPU (A100-40GB+).
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    CS_OFFLOAD = total_gb < 40.0
    print(f"GPU memory: {total_gb:.1f} GB → cpu offload: {CS_OFFLOAD}")
else:
    CS_OFFLOAD = True

cs_config = {
    "model_ckpt": CS_MODEL_CKPT,
    "output_dir": CS_OUTPUT_DIR,
    "resolution": 1024,
    "num_denoising_steps": 50,
    "guidance_scale": 3.5,
    "device": DEVICE,
    "dtype": DTYPE,
    "offload": CS_OFFLOAD,
    "fixed_layer": CS_FIXED_LAYER,
    # scenarios = prompts × seeds. Same prompt/diff seed => generation-to-generation
    # jitter; different prompt => content dependence of the top channels.
    "prompts": [
        "a red bicycle leaning against a brick wall",
        "a golden retriever sitting on a wooden dock at sunset",
        "a bowl of fresh strawberries on a marble countertop",
        "an astronaut planting a flag on the moon",
    ],
    "seeds": [0, 1, 2],
    "top_n": 20,
    "agg_k": 10,
    "n_channel_maps": 5,
    "n_control_maps": 0,  # set >0 to also dump low-ranked control channel maps
    # "representative_scenarios": [0, 3, 6],  # default: first 3
}

CS_CONFIG_PATH = "configs/channel_stability_colab.yaml"
with open(CS_CONFIG_PATH, "w") as fh:
    yaml.safe_dump(cs_config, fh, sort_keys=False)

print(open(CS_CONFIG_PATH).read())
print(
    f"scenarios: {len(cs_config['prompts'])} prompts × {len(cs_config['seeds'])} seeds "
    f"= {len(cs_config['prompts']) * len(cs_config['seeds'])}"
)

## 10. Run the experiment

12 scenarios × 50 steps on the 12B model — budget GPU time (much longer with CPU offload).

In [ ]:
!python -m src.experiments.channel_stability --config {CS_CONFIG_PATH}

## 11. Channel-stability results

The `stability_summary.json` line is the numeric answer: if the top-channel identities are
prompt-driven, same-prompt/different-seed Jaccard should exceed different-prompt Jaccard.

In [ ]:
import glob
import json

import pandas as pd
from IPython.display import Image, display

summary = json.load(open(f"{CS_OUTPUT_DIR}/stability_summary.json"))
print(f"scenarios: {summary['n_scenarios']} | seeds: {summary['n_seeds']}")
for k, v in summary["per_k"].items():
    print(
        f"top-{k:>2}: {v['n_distinct_channels_used']:>3} distinct channels used | "
        f"Jaccard same-prompt/diff-seed={v['mean_jaccard_same_prompt_diff_seed']} "
        f"diff-prompt={v['mean_jaccard_diff_prompt']}"
    )

print("\nfirst rows of the ordered top-20 table:")
display(pd.read_csv(f"{CS_OUTPUT_DIR}/channel_stability_topk.csv").head(20))

overlap = f"{CS_OUTPUT_DIR}/stability_overlap.png"
if os.path.isfile(overlap):
    display(Image(filename=overlap))

for sdir in sorted(glob.glob(f"{CS_OUTPUT_DIR}/scenarios/scenario_*")):
    print(sdir)
    display(Image(filename=f"{sdir}/image.png", width=256))
    pngs = sorted(glob.glob(f"{sdir}/channel_rank*.png")) + sorted(
        glob.glob(f"{sdir}/aggregated_*.png")
    )
    for png in pngs:
        display(Image(filename=png, width=200))

## 12. (Optional) Download outputs

Only needed if you didn't mount Drive in section 2 — otherwise outputs already live in
your Drive. Zips the Part 1 outputs and the Part 2 channel-stability dir (not the large
activation cache).

In [ ]:
if not USE_DRIVE:
    import shutil

    from google.colab import files

    dirs = [("figure3_outputs", OUTPUT_DIR)]
    if "CS_OUTPUT_DIR" in globals():
        dirs.append(("channel_stability", CS_OUTPUT_DIR))
    for name, d in dirs:
        if os.path.isdir(d) and os.listdir(d):
            archive = shutil.make_archive(f"/content/{name}", "zip", d)
            files.download(archive)
else:
    print("Outputs are already in Google Drive under:", DRIVE_ROOT)